# Java → C# Evaluation: Base Model vs Fine-tuned (`shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2`)

**Purpose:** Evaluate the fine-tuned model already on Hugging Face against the untuned base model
on the same Java→C# test dataset — **no training or adapter creation of any kind**.

| Item | Value |
|------|-------|
| Fine-tuned model | `shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2` |
| Base model | `unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit` |
| Dataset | CodeXGLUE Java↔C# (with XLCoST / CodeTransOcean fallback) |
| Metrics | BLEU · CodeBLEU · Exact Match · Syntax Accuracy · AST Similarity · Compilation Rate |

**Notebook layout**

1. Install Dependencies  
2. Imports  
3. Configuration  
4. Logging & Utilities  
5. Dataset Loading  
6. Prompt Builders (shared prompt format from training)  
7. Evaluation Metric Helpers  
8. Load Base Model → Run Inference → Evaluate  
9. Load Fine-tuned Model → Run Inference → Evaluate  
10. Base vs Fine-tuned Comparison Table  
11. Save Predictions & Reports

> Run top-to-bottom from a fresh kernel. No training cells exist in this notebook.

## 1. Install Dependencies

Install the full evaluation stack. `unsloth` is used to load both models efficiently in 4-bit.
`--quiet` keeps output manageable; re-run without it if you need to debug version conflicts.

In [ ]:
# Unsloth + HuggingFace stack (needed for 4-bit model loading).
!pip install -q unsloth transformers datasets accelerate peft bitsandbytes

# Evaluation metrics.
!pip install -q sacrebleu codebleu==0.7.0 code-bert-score

# tree-sitter grammars — pinned ABI so both Java and C# parsers are ABI-compatible.
!pip install -q tree-sitter==0.22.3 tree-sitter-java==0.21.0 tree-sitter-c-sharp==0.21.0 --force-reinstall

print("All dependencies installed.")

## 2. Imports

In [ ]:
import gc
import json
import logging
import math
import os
import random
import re
import shutil
import subprocess
import tempfile
from collections import Counter
from datetime import datetime
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from datasets import load_dataset

print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()}")

## 3. Configuration

Edit only this cell to change model IDs, dataset sizes, or output paths.

In [ ]:
# ---------------------------------------------------------------------------
# All tunable knobs live here. Nothing else needs editing.
# ---------------------------------------------------------------------------

# ---- Models ----
# Fine-tuned model (already uploaded to the Hub — loaded directly, no training).
FINETUNED_MODEL_ID = "shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2"

# Original pretrained base model used before fine-tuning.
BASE_MODEL_ID = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"

# Maximum sequence length (must match what was used during fine-tuning).
MAX_SEQ_LEN = 1024

# ---- Evaluation ----
# Number of Java/C# test pairs to evaluate. Increase for a more thorough run.
EVAL_N = 50

# ---- Reproducibility ----
SEED = 3407

# ---- Output ----
# Directory where prediction JSON files and the report will be written.
OUTPUT_DIR = "./eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"DEVICE={DEVICE} | EVAL_N={EVAL_N} | timestamp={TIMESTAMP}")
print(f"Fine-tuned : {FINETUNED_MODEL_ID}")
print(f"Base       : {BASE_MODEL_ID}")

## 4. Logging & Utilities

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("EvalOnly")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def log_gpu(tag: str = "") -> None:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        logger.info("GPU[%s] %.2f/%.2f/%.1f GB (alloc/reserved/total)",
                    tag, alloc, reserved, total)
    else:
        logger.warning("CUDA unavailable — running on CPU.")


def free_mem(*names: str) -> None:
    scope = globals()
    for n in names:
        if n in scope:
            del scope[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log_gpu("after free")


set_seed(SEED)
log_gpu("startup")

## 5. Dataset Loading (Java ↔ C#)

Loads the same dataset used during training (XLCoST → CodeTransOcean → CodeXGLUE fallback chain).
The test/validation pairs are extracted and capped at `EVAL_N`.

In [ ]:
# ---------------------------------------------------------------------------
# Exact same dataset loader used in the training notebook (nl-java-c-final-working).
# Tries XLCoST, then CodeTransOcean, then CodeXGLUE as the guaranteed fallback.
# ---------------------------------------------------------------------------

DATASET_CANDIDATES = [
    ("XLCoST",
     dict(path="codeparrot/xlcost-text-to-code", name="Csharp-program-level"),
     None, None),
    ("CodeTransOcean",
     dict(path="WeixiangYan/CodeTransOcean", name="MultilingualTrans"),
     "java", "csharp"),
    ("CodeXGLUE-Java-CS",
     dict(path="google/code_x_glue_cc_code_to_code_trans"),
     "java", "cs"),
]


def _normalize_split(ds, java_col, csharp_col):
    cols = set(ds.column_names)
    if java_col is None or java_col not in cols:
        java_col = next(
            (c for c in cols if c.lower() in ("java", "java_code", "src", "source")), None)
    if csharp_col is None or csharp_col not in cols:
        csharp_col = next(
            (c for c in cols if c.lower() in ("cs", "csharp", "c#", "cs_code", "tgt", "target")), None)
    if not java_col or not csharp_col:
        return None
    pairs = [{"java": j, "cs": c}
             for j, c in zip(ds[java_col], ds[csharp_col])
             if isinstance(j, str) and isinstance(c, str) and j.strip() and c.strip()]
    return pairs or None


def load_java_csharp_pairs():
    for label, kwargs, jcol, ccol in DATASET_CANDIDATES:
        try:
            logger.info("Trying dataset: %s ...", label)
            raw = load_dataset(**kwargs)
            splits = list(raw.keys()) if hasattr(raw, "keys") else ["train"]
            collected = {}
            for split in splits:
                pairs = _normalize_split(raw[split], jcol, ccol)
                if pairs:
                    collected[split] = pairs
            if collected:
                total = sum(len(v) for v in collected.values())
                logger.info("Loaded %s: %d pairs across %s",
                            label, total, list(collected))
                return label, collected
            logger.warning("%s: no usable Java/C# columns, skipping.", label)
        except Exception as exc:
            logger.warning("%s unavailable (%s). Trying next ...", label, exc)
    raise RuntimeError("No Java<->C# dataset loaded from any candidate.")


DATASET_LABEL, raw_pairs = load_java_csharp_pairs()

# Flatten all splits and apply the same quality filter used during training.
all_pairs = []
for split_pairs in raw_pairs.values():
    all_pairs.extend(split_pairs)


def is_valid_pair(java_code: str, csharp_code: str) -> bool:
    j, c = java_code.strip(), csharp_code.strip()
    if not j or not c or len(j) < 10 or len(c) < 10:
        return False
    if not any(t in j for t in (";", "{", "}")):
        return False
    if not any(t in c for t in (";", "{", "}")):
        return False
    return True


# De-duplicate and filter.
seen = set()
clean_pairs = []
for p in all_pairs:
    if not is_valid_pair(p["java"], p["cs"]):
        continue
    key = (p["java"].strip(), p["cs"].strip())
    if key in seen:
        continue
    seen.add(key)
    clean_pairs.append({"java": p["java"].strip(), "cs": p["cs"].strip()})

random.shuffle(clean_pairs)

# Use the last portion as "test" pairs (deterministic with SEED already set).
eval_pairs = clean_pairs[-min(EVAL_N, len(clean_pairs)):]

logger.info("Dataset: %s | total clean pairs: %d | eval subset: %d",
            DATASET_LABEL, len(clean_pairs), len(eval_pairs))
print(f"Sample Java input:\n{eval_pairs[0]['java'][:300]}\n")
print(f"Sample C# reference:\n{eval_pairs[0]['cs'][:300]}")

## 6. Prompt Builders (identical to training format)

These functions are copied verbatim from the training notebook so the inference prompt format
is **guaranteed** to match what the fine-tuned model saw during training.

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2 prompt builders — copied verbatim from nl-java-c-final-working.ipynb
# (Section 15) so the inference format is identical to the training format.
# DO NOT modify these unless you retrained with a different template.
# ---------------------------------------------------------------------------

STAGE2_INSTRUCTION = "Translate the following Java code into equivalent C#."
STAGE2_RESPONSE_MARKER = "### Response\n"
STAGE2_LANG_HINT = " Write the solution in C#."


def add_csharp_hint(instruction: str) -> str:
    instruction = instruction.strip()
    lowered = instruction.lower()
    if "c#" in lowered or "csharp" in lowered:
        return instruction
    return f"{instruction}{STAGE2_LANG_HINT}"


def build_stage2_inference_prompt(java_code: str) -> str:
    return (
        "### Instruction\n"
        f"{add_csharp_hint(STAGE2_INSTRUCTION)}\n\n"
        "### Java\n"
        f"{java_code.strip()}\n\n"
        "### Response\n"
    )


# Quick sanity-check
_sample = build_stage2_inference_prompt(
    "public int add(int a, int b) { return a + b; }")
print(_sample)

## 7. Evaluation Metric Helpers

Copied verbatim from `nl-java-c-final-working.ipynb` (Section 19) — no modification.

Metrics computed:
- **BLEU** (sacrebleu corpus BLEU)
- **CodeBLEU** (tree-sitter C# parser)
- **Exact Match** (whitespace-normalised)
- **Syntax Accuracy** (fraction that parse without errors)
- **AST Similarity** (bag-of-node-types cosine)
- **Compilation Rate** (real `dotnet`/`csc`/`mcs` if available, else tree-sitter proxy)

In [ ]:
import sacrebleu
from codebleu import calc_codebleu


def _normalize_code(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())


# ---- tree-sitter C# parser (best-effort) ------------------------------------
def get_csharp_parser():
    try:
        from tree_sitter import Language, Parser
        import tree_sitter_c_sharp as tscs
        lang = Language(tscs.language())
        try:
            return Parser(lang)
        except TypeError:
            p = Parser()
            p.set_language(lang)
            return p
    except Exception as exc:
        print(f"C# tree-sitter parser unavailable: {exc}")
        return None


CS_PARSER = get_csharp_parser()


def _collect_node_types(node, acc):
    acc.append(node.type)
    for child in node.children:
        _collect_node_types(child, acc)


def _ast_node_types(code: str):
    if CS_PARSER is None:
        return None
    try:
        tree = CS_PARSER.parse(bytes(code, "utf8"))
        acc = []
        _collect_node_types(tree.root_node, acc)
        return acc
    except Exception:
        return None


def _parse_is_valid(code: str):
    if CS_PARSER is None:
        return None
    try:
        tree = CS_PARSER.parse(bytes(code, "utf8"))
        return not tree.root_node.has_error
    except Exception:
        return False


def _bag_cosine(a, b):
    ca, cb = Counter(a), Counter(b)
    keys = set(ca) | set(cb)
    dot = sum(ca[k] * cb[k] for k in keys)
    na = math.sqrt(sum(v * v for v in ca.values()))
    nb = math.sqrt(sum(v * v for v in cb.values()))
    return dot / (na * nb) if na and nb else 0.0


# ---- Compilation rate -------------------------------------------------------
def _find_csharp_compiler():
    for tool in ("dotnet", "csc", "mcs"):
        if shutil.which(tool):
            return tool
    return None


CSHARP_COMPILER = _find_csharp_compiler()
logger.info("C# compiler: %s",
            CSHARP_COMPILER or "none (syntax proxy will be used)")


def _wrap_csharp(code: str) -> str:
    if re.search(r"\b(class|struct|interface|enum|namespace)\b", code):
        return code
    return ("using System;\nusing System.Collections.Generic;\n"
            "public class Wrapper {\n" + code + "\n}\n")


def _compiles_with_compiler(code: str) -> bool:
    with tempfile.TemporaryDirectory() as tmp:
        src = os.path.join(tmp, "Program.cs")
        with open(src, "w", encoding="utf-8") as fh:
            fh.write(_wrap_csharp(code))
        try:
            result = subprocess.run(
                [CSHARP_COMPILER, "-target:library", src],
                capture_output=True, timeout=30)
            return result.returncode == 0
        except Exception:
            return False


def compilation_rate(preds: List[str]):
    if CSHARP_COMPILER in ("csc", "mcs"):
        oks = [_compiles_with_compiler(p) for p in preds]
        return sum(oks) / len(preds), f"real-compiler({CSHARP_COMPILER})"
    valids = [v for v in (_parse_is_valid(p) for p in preds) if v is not None]
    if not valids:
        return None, "unavailable"
    return sum(valids) / len(valids), "syntax-proxy(tree-sitter)"


# ---- Main evaluation function -----------------------------------------------
def evaluate_csharp(predictions: List[str], references: List[str]) -> Dict:
    report: Dict = {}

    # BLEU
    report["BLEU"] = round(
        sacrebleu.corpus_bleu(predictions, [references]).score, 4)

    # CodeBLEU
    try:
        cb = calc_codebleu(references, predictions, lang="c_sharp")
        report["CodeBLEU"] = {k: round(v, 4) for k, v in cb.items()}
    except Exception as exc:
        report["CodeBLEU"] = {"error": str(exc)}

    # Exact match
    report["ExactMatch"] = round(
        sum(_normalize_code(p) == _normalize_code(r)
            for p, r in zip(predictions, references)) / len(predictions), 4)

    # Syntax accuracy
    syn = [v for v in (_parse_is_valid(p)
                       for p in predictions) if v is not None]
    report["SyntaxAccuracy"] = round(sum(syn) / len(syn), 4) if syn else None

    # AST similarity
    sims = []
    for p, r in zip(predictions, references):
        a, b = _ast_node_types(p), _ast_node_types(r)
        if a is not None and b is not None:
            sims.append(_bag_cosine(a, b))
    report["AST_Similarity"] = round(
        sum(sims) / len(sims), 4) if sims else None

    # Compilation rate
    rate, method = compilation_rate(predictions)
    report["CompilationRate"] = {
        "value": round(rate, 4) if rate is not None else None,
        "method": method,
    }
    return report


print("Evaluation helpers ready. CS_PARSER:",
      "OK" if CS_PARSER else "unavailable")

## 8. Shared Inference Helper

A model-agnostic generation function reused for both the base and fine-tuned models.

In [ ]:
from unsloth import FastLanguageModel


@torch.inference_mode()
def _run_generation(gen_model, gen_tokenizer, prompt_text: str,
                    max_new_tokens: int = 400) -> str:
    """Greedy generation — identical to the helper in nl-java-c-final-working."""
    FastLanguageModel.for_inference(gen_model)
    inputs = gen_tokenizer(
        prompt_text, return_tensors="pt").to(gen_model.device)
    output = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=gen_tokenizer.eos_token_id,
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)


def generate_csharp_predictions(gen_model, gen_tokenizer,
                                pairs: List[Dict],
                                max_new_tokens: int = 400,
                                label: str = "") -> Tuple[List[str], List[str]]:
    """Generate C# predictions for a list of {'java', 'cs'} pairs.

    Uses the same prompt builder as training so the format is never mismatched.
    Returns (predictions, references).
    """
    preds, refs = [], []
    for i, p in enumerate(pairs):
        prompt = build_stage2_inference_prompt(p["java"])
        try:
            decoded = _run_generation(
                gen_model, gen_tokenizer, prompt, max_new_tokens)
            pred = decoded.split(STAGE2_RESPONSE_MARKER)[-1].strip()
        except Exception:
            logger.exception(
                "[%s] Generation failed at index %d; using empty string.", label, i)
            pred = ""
        preds.append(pred)
        refs.append(p["cs"])
        if (i + 1) % 10 == 0:
            logger.info("[%s] %d / %d generated.", label, i + 1, len(pairs))
    return preds, refs


print("generate_csharp_predictions() ready.")

## 9. Base Model — Load → Infer → Evaluate

Loads the **untuned** base model (`unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit`), runs
inference on all `eval_pairs`, and evaluates with `evaluate_csharp()`.  
The model is freed from VRAM immediately after evaluation.

In [ ]:
# ----- Load base model -------------------------------------------------------
logger.info("Loading base model: %s", BASE_MODEL_ID)
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
log_gpu("after base model load")

# ----- Inference -------------------------------------------------------------
logger.info("Running base model inference on %d pairs ...", len(eval_pairs))
base_predictions, base_references = generate_csharp_predictions(
    base_model, base_tokenizer, eval_pairs, label="base")

# ----- Evaluate --------------------------------------------------------------
logger.info("Evaluating base model ...")
base_report = evaluate_csharp(base_predictions, base_references)

print("\n===== Base Model Evaluation Report =====")
print(f"Model : {BASE_MODEL_ID}")
print(f"Pairs : {len(eval_pairs)}")
print(json.dumps(base_report, indent=2))

# ----- Free VRAM -------------------------------------------------------------
free_mem("base_model", "base_tokenizer")

## 10. Fine-tuned Model — Load → Infer → Evaluate

Loads the **fine-tuned** model directly from the Hugging Face Hub
(`shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2`) using the same 4-bit loading path.  
No LoRA adapter creation or training happens here.

In [ ]:
# ----- Load fine-tuned model directly from HF Hub ---------------------------
logger.info("Loading fine-tuned model from Hub: %s", FINETUNED_MODEL_ID)
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNED_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
log_gpu("after fine-tuned model load")

# ----- Inference -------------------------------------------------------------
logger.info("Running fine-tuned model inference on %d pairs ...",
            len(eval_pairs))
ft_predictions, ft_references = generate_csharp_predictions(
    ft_model, ft_tokenizer, eval_pairs, label="fine-tuned")

# Sanity check: references must be identical to base model run.
assert ft_references == base_references, "Reference mismatch — eval_pairs must not change between runs."

# ----- Evaluate --------------------------------------------------------------
logger.info("Evaluating fine-tuned model ...")
ft_report = evaluate_csharp(ft_predictions, ft_references)

print("\n===== Fine-tuned Model Evaluation Report =====")
print(f"Model : {FINETUNED_MODEL_ID}")
print(f"Pairs : {len(eval_pairs)}")
print(json.dumps(ft_report, indent=2))

# ----- Free VRAM -------------------------------------------------------------
free_mem("ft_model", "ft_tokenizer")

## 11. Base vs Fine-tuned Comparison Table

Side-by-side metrics with absolute improvement (Fine-tuned − Base).

In [ ]:
# ---------------------------------------------------------------------------
# Comparison table: Base vs Fine-tuned (Java → C#)
# ---------------------------------------------------------------------------

def _get_bleu(rpt):
    return rpt.get("BLEU", float("nan"))


def _get_codebleu(rpt):
    cb = rpt.get("CodeBLEU", {})
    return cb.get("codebleu", float("nan")) if isinstance(cb, dict) else float("nan")


def _get(rpt, key):
    v = rpt.get(key)
    return float("nan") if v is None else float(v)


def _get_compilation(rpt):
    cr = rpt.get("CompilationRate", {})
    if isinstance(cr, dict):
        v = cr.get("value")
        return float("nan") if v is None else float(v)
    return float("nan")


def _fmt(v):
    return f"{v:.4f}" if not (isinstance(v, float) and math.isnan(v)) else "N/A"


def _diff(ft_v, base_v):
    if any(isinstance(x, float) and math.isnan(x) for x in (ft_v, base_v)):
        return "N/A"
    diff = ft_v - base_v
    return (f"+{diff:.4f}" if diff >= 0 else f"{diff:.4f}")


rows = [
    ("BLEU",             _get_bleu(base_report),         _get_bleu(ft_report)),
    ("CodeBLEU",         _get_codebleu(base_report),     _get_codebleu(ft_report)),
    ("Exact Match",      _get(base_report, "ExactMatch"),
     _get(ft_report, "ExactMatch")),
    ("Syntax Accuracy",  _get(base_report, "SyntaxAccuracy"),
     _get(ft_report, "SyntaxAccuracy")),
    ("AST Similarity",   _get(base_report, "AST_Similarity"),
     _get(ft_report, "AST_Similarity")),
    ("Compilation Rate", _get_compilation(
        base_report),  _get_compilation(ft_report)),
]

W = 26
DIVIDER = "=" * 74
print(f"\n{DIVIDER}")
print(f"  Java → C#  │  Base vs Fine-tuned Comparison")
print(DIVIDER)
print(f"  {'Metric':{W}}  {'Base Model':>12}  {'Fine-tuned':>12}  {'Improvement':>12}")
print(f"  {'-'*W}  {'-'*12}  {'-'*12}  {'-'*12}")
for name, bv, fv in rows:
    print(f"  {name:{W}}  {_fmt(bv):>12}  {_fmt(fv):>12}  {_diff(fv, bv):>12}")
print(DIVIDER)
print(f"\n  Dataset   : {DATASET_LABEL}")
print(f"  Eval pairs: {len(eval_pairs)}")
print(f"  Base      : {BASE_MODEL_ID}")
print(f"  Fine-tuned: {FINETUNED_MODEL_ID}")

# Also display compilation method used
cr_base = base_report.get("CompilationRate", {})
cr_ft = ft_report.get("CompilationRate", {})
if isinstance(cr_base, dict):
    print(f"\n  Compilation method: {cr_base.get('method', 'N/A')}")

## 12. Save Predictions & Reports

Writes predictions and metrics to `OUTPUT_DIR` so the run is fully reproducible.

In [ ]:
# ---------------------------------------------------------------------------
# Persist everything to OUTPUT_DIR so the results can be inspected later.
# ---------------------------------------------------------------------------

def _save_json(obj, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    logger.info("Saved %s", path)
    return path


# 1) Base model predictions (separate file — never overwrites fine-tuned).
_save_json(
    [{"java": p["java"], "cs_reference": p["cs"], "cs_base": pred}
     for p, pred in zip(eval_pairs, base_predictions)],
    f"base_predictions_{TIMESTAMP}.json",
)

# 2) Fine-tuned model predictions.
_save_json(
    [{"java": p["java"], "cs_reference": p["cs"], "cs_finetuned": pred}
     for p, pred in zip(eval_pairs, ft_predictions)],
    f"finetuned_predictions_{TIMESTAMP}.json",
)

# 3) Combined metrics report.
combined_report = {
    "timestamp": TIMESTAMP,
    "dataset": DATASET_LABEL,
    "eval_pairs": len(eval_pairs),
    "base_model": BASE_MODEL_ID,
    "finetuned_model": FINETUNED_MODEL_ID,
    "base_metrics": base_report,
    "finetuned_metrics": ft_report,
    "comparison": {
        name: {"base": _fmt(bv), "finetuned": _fmt(fv),
               "improvement": _diff(fv, bv)}
        for name, bv, fv in rows
    },
}
_save_json(combined_report, f"eval_report_{TIMESTAMP}.json")

print(f"\nAll outputs written to: {os.path.abspath(OUTPUT_DIR)}")